# Optimized (As much as possible) GARCH Model for Volatility Forecasting

The GARCH Model is used to:
- Model volatility clustering in financial time series.
- Forecast future volatility.
- Optimize trading strategies using historical price movements.

This notebook fixes common mistakes in GARCH modeling, ensuring robust financial analysis.



In [ ]:
#IMPORT MODULES     

import arch
from arch import arch_model
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import itertools
import numpy as np

Data Preprocessing
We load financial time series data, compute returns, and clean missing values.

In [ ]:
df = pd.read_excel("/Users/atulramesh/Desktop/data2.xlsx")
df['Dates'] = pd.to_datetime(df['Dates'], errors='coerce') #Convert to DateTime Format
df.set_index('Dates', inplace=True) 

#LOAD RETURNS ON TO ANOTHER COLUMNS
df['Returns'] = df['PX_OPEN'].pct_change() #put interms of percentage change for returns
df.dropna(inplace=True) #Remove Null value rows

ACF PACF ANALYSIS to analyse ACF (AutoCorrelation Function) and PACF (Partial AutoCorrelation Function) 

In [ ]:
#ACF PACF ANALYSIS
num_lags = min(40, len(df)-1) #set num_lags to optimise the number of lags
fig, axes = plt.subplots(1, 2, figsize=(12,6)) #set plot
sm.graphics.tsa.plot_acf(df['Returns']**2, lags=num_lags, ax=axes[0]) #**2 to input squared returns for ACF 
sm.graphics.tsa.plot_pacf(df['Returns']**2, lags=num_lags, ax=axes[1]) #**2 to input squared returns for PACF
plt.show() #plot acf and pacf

Optimising the Hyperparameters, being p,q for the GARCH MODEL

In [ ]:
#init best_aic and p,q
best_aic = float("inf")
best_pq = None 

#Grid search for best p,q values using iteration
for p, q in itertools.product(range(1, 3), range(1, 3)):  # Testing (1,1) to (2,2)
    model = arch_model(df['Returns'], p=p, q=q, mean="AR", vol="GARCH", dist="t")
    result = model.fit(disp="off")
    
    # Check If AIC is Lower (Better Fit)
    if result.aic < best_aic:
        best_aic = result.aic
        best_pq = (p, q)

#Show best model selection
print(f"Best GARCH Order: {best_pq} with AIC: {best_aic}")

Fitting the best GARCH Model
OPTIMIZATIONS
inputting the best p,q
use t tail dist as market returns are fat tailed such as cauchy
use mean = AR as you are doing simple returns not volatility modelling, use HARX if you are doing volatility modelling

In [ ]:
#Best model fit
GARCH_Model = arch_model(df['Returns'], p=best_pq[0], q=best_pq[1], mean="AR", vol="GARCH", dist="t") # T- TAIL TO HANDLE FAT TAILS (MARKET SHOCKS) SUCH AS IN CAUCHY AND USE MEAN = AR FOR RETURNS MODELLING
#IF YOU NEED FOR VOLATILITY FORECAST USE HARX IN MEAN
gm_result = GARCH_Model.fit(update_freq=1)

#print the summary
print(gm_result.summary())

Residual Diagnostics   
Check if residuals show autocorrelation, if not model is misspecifed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,6)) #set plot
sm.graphics.tsa.plot_acf(gm_result.resid, lags=30, ax=axes[0]) #residual plot for ACF
sm.graphics.tsa.plot_pacf(gm_result.resid, lags=30, ax=axes[1]) #residual plot for PACF
plt.show() #plot

VOLATILITY FORECAST using dynamic forecast horizon 
ensure that forecast horizon is at most a week or 10% the timeframe of your data 

In [ ]:
horizon = min(7, len(df) // 10) #set forecast to minimum a week or maximum 10% of timeframe


gm_forecast = gm_result.forecast(horizon=horizon) #forecast the volatility

print(gm_forecast.variance) #print the forecasted variance

Model Evaluation

AIC  - Akailike Likelyhood Comparison (Lower the Better)
Log Likelyhood - Higher the better
BIC - Bayesian Information Criterion (Helps Prevent Overfitting)

In [ ]:
print(f"Log-Likelihood: {gm_result.loglikelihood}")
print(f"AIC: {gm_result.aic}")
print(f"BIC: {gm_result.bic}")

All in all this code helps you implement a GARCH with your own financial data (I used returns over 10 years using Bloomberg)
Thank you